# 实验五：使用 PyTorch 训练 MNIST 手写数字分类模型

在实验四中，我们基于 `numpy` 实现了手动的前向传播、BP 和 梯度下降。现在我们将使用完整的 Pytorch 完整工具包来实现一次更具体的实验。在这个框架下，我们不用再考虑算子上的内容，而是可以将更多的精力放在**数据、训练流程、验证与实验结论**。这也是我们第一次动手处理数据集，并实现一个较为实际的问题。

这里我们用**全连接模型**而不是 CNN 来处理 MNIST 的二维图像，主要是为了和上次的手动 Numpy 进行衔接，避免在引入 pytorch 的同时给同学们带来过多的知识点，难度上会更平滑一点。

实验主线：认识数据 → 建立分类模型 → 检查一次更新 → 小批量训练 → 验证选模 → 最终测试与错误分析。

**Q1–Q7 必做；Q8 为选做学习率对照。** Q1 是教师演示。先写预测，再补全代码，最后用输出解释结果。


## 0 · 文件、环境和实验约定

当前市面上的主流 CPU 足以应对这次实验，不需要调用 GPU 算力，也不会产生过高的能耗。

实验设计与内容参考 [d2l相关章节](https://zh.d2l.ai/chapter_linear-networks/softmax-regression-scratch.html)。但不需要安装 `d2l` 的包。

第一次联网运行时，将下一单元的 `DOWNLOAD` 改为 `True`；数据缓存就绪后改回 `False`。离线时复制教师已准备的 `data/MNIST/raw/` 目录；这里的下载开关与 pip 软件源无关。

| 设置       | 本版约定                                                                |
| ---------- | ----------------------------------------------------------------------- |
| 数据划分   | 官方训练集固定拆成 50,000 训练、10,000 验证；官方测试集 10,000 最终使用 |
| 输入       | float32，单通道 28×28，像素范围 [0,1]；无额外标准化或增强               |
| 标签       | long，类别下标 0–9，形状 (B,)                                           |
| 主模型     | Flatten → Linear(784,128) → ReLU → Linear(128,10)                       |
| 输出与损失 | logits，不加 softmax；CrossEntropyLoss 默认 mean                        |
| 优化       | SGD，无动量；lr=0.1，batch_size=128，epochs=5                           |
| 随机设置   | 划分、初始化、批次顺序分别固定；CPU 为默认验证环境                      |
| 选模规则   | 验证准确率优先，相同准确率时验证损失较小优先，完全相同取较早轮次        |

先预测：框架计算了梯度，是否意味着数据、损失和训练流程就一定正确？


In [ ]:
import math
import os
from pathlib import Path
import sys
import time

import numpy as np
import platform
import torch
from torch import nn
from torch.utils.data import TensorDataset, Subset
import torchvision

from lab_utils import (
    seed_everything,
    data_root,
    load_train_val,
    load_test,
    make_loader,
    snapshot,
    plot_samples,
    plot_history,
    collect_errors,
)

assert Path("lab_utils.py").exists(), "请在实验五目录启动 notebook"

SEED, SPLIT_SEED, ORDER_SEED = 42, 42, 2026
BATCH_SIZE, EPOCHS, LR = 128, 5, 0.1
DOWNLOAD = False  # 第一次联网下载时改为 True，缓存就绪后改回 False。
DEVICE = torch.device("cpu")  # 已有 CUDA 环境可自行改为 torch.device("cuda")。
# 小模型使用过多 CPU 线程可能更慢；限制线程数便于普通电脑复现。
torch.set_num_threads(min(4, os.cpu_count() or 1))
seed_everything(SEED)
print("Python:", sys.version.split()[0], sys.executable)
print("PyTorch:", torch.__version__, "torchvision:", torchvision.__version__)
print(
    "device:",
    DEVICE,
    "threads:",
    torch.get_num_threads(),
    "machine:",
    platform.machine(),
)
print("data root:", data_root().resolve())
print(
    {
        "seed": SEED,
        "split_seed": SPLIT_SEED,
        "order_seed": ORDER_SEED,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "epochs": EPOCHS,
    }
)

## Q1 · 从手写 BP 到自动求导（必做，教师演示）

在实验四中，我们设定的模型为：$Z_1=XW_1+b_1$，$A_1=\tanh(Z_1)$，$\hat y=A_1W_2+b_2$，$L=\mathrm{mean}((\hat y-y)^2)$。

下面是一个已经完成的代码块，前半部分为上次实验中用 `numpy` 实现的内容，后半部分则是用 `torch` 实现的内容。请先阅读代码，了解并感受 `torch` 在这个人物上的简洁之处，以及每层之间的信息是怎么传递的。

> 尽量保证能够独立说出每一行代码在做什么事，再往后推进。相关注释已经在代码中补充。


In [ ]:
# --- Numpy 实现的 BP 算法 ---
rng = np.random.default_rng(7)
x_np = np.array([[-0.7], [0.2], [0.9]], dtype=np.float64)
y_np = np.sin(x_np)
p_np = {
    "W1": rng.normal(size=(1, 3)),
    "b1": np.zeros((1, 3)),
    "W2": rng.normal(scale=1 / np.sqrt(3), size=(3, 1)),
    "b2": np.zeros((1, 1)),
}
a = np.tanh(x_np @ p_np["W1"] + p_np["b1"])
yhat = a @ p_np["W2"] + p_np["b2"]
dy = 2 * (yhat - y_np) / len(x_np)
dz = (dy @ p_np["W2"].T) * (1 - a**2)
manual = {
    "W1": x_np.T @ dz,
    "b1": dz.sum(0, keepdims=True),
    "W2": a.T @ dy,
    "b2": dy.sum(0, keepdims=True),
}

# --- torch 实现的 BP 算法 ---
# 这里的参数用字典的形式存储，key 和 p_np 中的索引相同，但存储的内容变成了 Tensor，即张量。
params = {
    k: nn.Parameter(torch.tensor(v, dtype=torch.float64)) for k, v in p_np.items()
}
# 值得说明的是，这里的 tensor 初始化传入的是 numpy 的 array，
# 初始化的过程与方式已经打包进了工具包，这里是直接将 NumPy 数组转成了同形状的 Tensor。
x_demo, y_demo = torch.tensor(x_np), torch.tensor(y_np)


def demo_loss():
    """使用 torch 计算得到的损失。"""
    Z1 = x_demo @ params["W1"] + params["b1"]
    A1 = torch.tanh(Z1)
    y_pred = A1 @ params["W2"] + params["b2"]
    return ((y_pred - y_demo) ** 2).mean()


before = {k: p.detach().clone() for k, p in params.items()}

# 初始化 torch 的优化器，并设置好学习率 lr
optimizer_demo = torch.optim.SGD(params.values(), lr=0.001)
optimizer_demo.zero_grad(set_to_none=True)  # 梯度清零

demo_loss().backward()
first_grad = {k: p.grad.clone() for k, p in params.items()}
for k, p in params.items():
    np.testing.assert_allclose(p.grad.numpy(), manual[k], rtol=1e-10, atol=1e-12)
    assert torch.equal(p, before[k])
    print(k, "max gradient error =", np.max(np.abs(p.grad.numpy() - manual[k])))


## Q2 · 数据、标签与小批量（必做）

教师提供固定划分和加载工具。此时只使用官方训练数据；测试集到 Q7 才加载和评估。

查看样本并补全形状表，说明每个维度的含义。为什么标签是整数类别下标，而不是待拟合的连续数值？

| 量                | 形状 | dtype | 含义 |
| ----------------- | ---- | ----- | ---- |
| 一个 batch 的图像 | 待填 | 待填  | 待填 |
| 一个 batch 的标签 | 待填 | 待填  | 待填 |
| 展平后图像        | 待填 | 待填  | 待填 |
| 模型输出          | 待填 | 待填  | 待填 |

**问题**：对于这里的图像，我们将其转换成了什么样的数据？

**回答**：TODO


In [ ]:
# 这里的数据集下载已经打包成 `load_train_val` 函数，只需要自行传参即可
train_set, val_set = load_train_val(download=DOWNLOAD, split_seed=SPLIT_SEED)
assert len(train_set) == 50000 and len(val_set) == 10000
assert not set(train_set.indices).intersection(val_set.indices)
assert len(set(train_set.indices) | set(val_set.indices)) == 60000

val_loader = make_loader(val_set, BATCH_SIZE)
train_eval_loader = make_loader(train_set, BATCH_SIZE)

# 探索样本使用独立 loader，不消耗正式训练的随机 generator。
x_batch, y_batch = next(
    iter(make_loader(train_set, BATCH_SIZE, shuffle=True, seed=ORDER_SEED))
)
assert x_batch.shape == (BATCH_SIZE, 1, 28, 28)
assert x_batch.dtype == torch.float32 and y_batch.dtype == torch.long
assert y_batch.shape == (BATCH_SIZE,) and 0 <= y_batch.min() <= y_batch.max() <= 9
assert 0 <= x_batch.min() <= x_batch.max() <= 1

print("images:", x_batch.shape, x_batch.dtype, "labels:", y_batch.shape, y_batch.dtype)
print(
    "batches/epoch:",
    math.ceil(len(train_set) / BATCH_SIZE),
    "last batch:",
    len(train_set) % BATCH_SIZE,
)


# 确认我们下载好的数据集
_ = plot_samples(x_batch[:12], y_batch[:12], title="Training samples")

## Q3 · 建立分类模型

实现函数 `build_model(kind="mlp", seed=42)`，并返回位于 `DEVICE` 的网络。留给定的种子设置。

`DEVICE` 制定计算设备（CPU/GPU）。本实验使用 CPU； `.to(DEVICE)` 函数是将模型参数或数据移动到制定设备。参与同一次计算的模型和数据需要位于同一设备。（统一设备上的数据容量就是所谓“带宽”）。

下面我们将设计两种网络：

- `mlp`：Flatten → Linear(784,128) → ReLU → Linear(128,10)。
- `linear`：Flatten → Linear(784,10)，供 Q6 对照。
- 未知 kind 应抛出 ValueError。

输出十个 logits（未经归一化的分数），不添加 softmax。


In [ ]:
def build_model(kind="mlp", seed=42):
    seed_everything(seed)

    # HINT：查阅 `nn.Sequential()` 的用法，中间每一层用 `nn.tensor()` 来管理输入与输出
    # TODO：按题意构造两种网络，未知 kind 抛出 ValueError。
    # TODO：最后将模型移动到 DEVICE；输出层直接返回十个 logits

    return model.to(DEVICE)

In [ ]:
model = build_model("mlp", SEED)
scores = model(x_batch.to(DEVICE))
assert scores.shape == (len(y_batch), 10)
assert torch.isfinite(scores).all()
assert sum(p.numel() for p in model.parameters()) == 101770
assert sum(p.numel() for p in build_model("linear", SEED).parameters()) == 7850
print(model)
print("MLP parameters:", sum(p.numel() for p in model.parameters()))
print("logits of first sample:", scores[0].detach().cpu())

## Q4 · 分类损失与一次参数更新（必做）

设第 i 个样本的类别分数为 $z_{i,k}$，真实类别为 $y_i$，交叉熵为

$$
L=-\frac1B\sum_i\log\frac{\exp(z_{i,y_i})}{\sum_{k=0}^9\exp(z_{i,k})}.
$$

实现中直接使用 `nn.CrossEntropyLoss()`，这是一个类，它接收 logits 和整数类别标签，并稳定地组合所需运算。

先运行下面的小例子：两个结果的预测类别相同，准确率相同，损失是否相同？为什么不能用 argmax 后的准确率直接反向传播？


In [ ]:
criterion = nn.CrossEntropyLoss()
target_demo = torch.tensor([3])
for margin in [0.2, 5.0]:
    z = torch.zeros(1, 10)
    z[0, 3] = margin
    print(
        f"margin={margin}, predicted={z.argmax(1).item()}, "
        f"p(true)={z.softmax(1)[0, 3].item():.4f}, CE={criterion(z, target_demo).item():.4f}"
    )

下面补全 `train_step`，返回本批**更新前**的损失总和、预测正确数和样本数。

固定接口为 `{"loss_sum": float, "correct": int, "n": int}`。损失函数采用默认 mean，因此乘以本批真实样本数得到 loss_sum。不要把带计算图的 loss 保存在训练历史中。

顺序：清除旧梯度 → 前向 → 损失 → backward → step。图像与标签均需移动到同一 device。


In [ ]:
def train_step(model, x, y, optimizer, criterion, device):
    """执行单步训练"""
    model.train()
    # TODO: 移动数据到指定设备上
    # TODO: 通过优化器清除所管理参数的旧梯度
    # TODO: 进行一次前向传播
    # TODO: 计算前向传播后的损失
    # TODO: 进行一次反向传播，这里应保证损失与梯度有限
    # TODO: 反向传播后更新参数
    # 以字典形式返回更新前的 loss_sum、correct 和 n。
    raise NotImplementedError("Q4：完成单批训练")

In [ ]:
# 用新模型做一次检查，之后正式训练再重新初始化。
step_model = build_model("mlp", SEED)
step_optimizer = torch.optim.SGD(step_model.parameters(), lr=LR)
params_before = snapshot(step_model)
step_metrics = train_step(
    step_model, x_batch, y_batch, step_optimizer, criterion, DEVICE
)
assert step_metrics["n"] == len(y_batch)
assert math.isfinite(step_metrics["loss_sum"])
assert 0 <= step_metrics["correct"] <= step_metrics["n"]
assert any(
    not torch.equal(params_before[k], v.cpu())
    for k, v in step_model.state_dict().items()
)
assert all(torch.isfinite(p).all() for p in step_model.parameters())
print("single batch before-update metrics:", step_metrics)
# 训练完后的同一批损失仅作观察；任意学习率下一步都下降不是必须满足的数学保证。
with torch.no_grad():
    print(
        "same-batch loss after update:",
        criterion(step_model(x_batch.to(DEVICE)), y_batch.to(DEVICE)).item(),
    )

## Q5 · 从一批到完整训练（必做）

实现 `evaluate` 和 `train_epoch`，接口均返回 `{"loss": float, "accuracy": float, "n": int}`。

- `evaluate`：eval 模式和 no_grad 上下文；不更新模型、不清理或改写已有梯度。
- `train_epoch`：遍历 loader，调用 Q4 的 train_step 并汇总；每个 batch 更新一次。
- 两者均按真实样本数加权：`sum(batch_mean_loss * batch_n) / total_n`，准确率为总正确数／总样本数。空 loader 应抛出 ValueError。

提示：

- `model.eval()` 设置 Dropout/BatchNorm 等模块的行为；
- `no_grad()` 关闭梯度记录。当前网络没有这些模块，仍应区分二者。

先用不相同大小 batch 的已知分数检查汇总，再做小样本拟合，最后进入完整训练。


In [ ]:
def evaluate(model, loader, criterion, device) -> dict:
    """衡量该模型处理的效果

    Args:
        model: 需要评估的模型
        loader: 数据加载器
        criterion: 评估类
        device: 用于计算的设备

    Returns:
        dict: 相关衡量指标的字典，包括 loss, accuracy, n
    """
    # TODO：将模型切换到评估模式
    loss_sum, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            # TODO：数据移动到相关设备上
            # TODO：输入的 x 过一次模型，得到 scores
            # TODO：计算一次损失
            loss = ...  # TODO：计算这里的损失
            if not torch.isfinite(loss):
                raise FloatingPointError("评估存在非有限损失")
            # TODO：更新 loss_sum, correct, n

    if n == 0:
        raise ValueError("不能评估空数据集")

    # TODO：返回 loss、accuracy、n
    raise NotImplementedError("Q5：完成评估")


def train_epoch(model, loader, optimizer, criterion, device) -> dict:
    """遍历训练集，完成一个 epoch 的训练

    Args:
        model: 需要训练的模型
        loader: 数据加载类
        optimizer: 优化类
        criterion: 评价类
        device: 计算单元

    Returns:
        dict: 训练结果的字典
    """
    loss_sum, correct, n = 0.0, 0, 0
    for x, y in loader:
        batch = train_step()  # TODO：填入相关参数
        loss_sum += batch["loss_sum"]
        correct += batch["correct"]
        n += batch["n"]

    if n == 0:
        raise ValueError("不能训练空数据集")

    raise NotImplementedError("Q5：完成一轮训练")

下面用五个已知分数，检查损失和准确率是否汇总正确。


In [ ]:
known_scores = torch.tensor(
    [[5.0, 0.0], [0.0, 5.0], [5.0, 0.0], [0.0, 5.0], [0.0, 5.0]]
)
known_y = torch.tensor([0, 1, 0, 1, 0])  # 最后一张故意错分且损失很大。
known_loader = make_loader(TensorDataset(known_scores, known_y), batch_size=2)
identity = nn.Identity().to(DEVICE)
metrics = evaluate(identity, known_loader, criterion, DEVICE)
expected_loss = criterion(known_scores, known_y).item()
assert abs(metrics["loss"] - expected_loss) < 1e-6
assert metrics["accuracy"] == 4 / 5 and metrics["n"] == 5
batch_means = [criterion(a, b).item() for a, b in known_loader]
wrong_mean = sum(batch_means) / len(batch_means)
assert abs(wrong_mean - expected_loss) > 0.1
print(
    "correct sample-weighted loss:",
    metrics["loss"],
    "wrong batch-weighted loss:",
    wrong_mean,
)

In [ ]:
# 固定 32 张训练图像反复拟合，用新模型；不使用验证或测试数据。
tiny_x, tiny_y = next(iter(make_loader(Subset(train_set, range(32)), batch_size=32)))
tiny_model = build_model("mlp", SEED)
tiny_optimizer = torch.optim.SGD(tiny_model.parameters(), lr=LR)
tiny_loader = make_loader(TensorDataset(tiny_x, tiny_y), batch_size=32)
tiny_before = evaluate(tiny_model, tiny_loader, criterion, DEVICE)
for _ in range(100):
    train_step(tiny_model, tiny_x, tiny_y, tiny_optimizer, criterion, DEVICE)
tiny_after = evaluate(tiny_model, tiny_loader, criterion, DEVICE)
print("tiny set before:", tiny_before, "after:", tiny_after)
assert tiny_after["loss"] < tiny_before["loss"] * 0.2, (
    "先排查小样本训练，暂不进入完整训练"
)

### 完整训练与验证选轮次

下面的 `fit` 是教师提供的组织代码，调用你实现的 train_epoch 和 evaluate。逐轮阅读并解释它如何保存最佳参数。

每个 epoch 结束后，用**同一组固定参数**重新评估训练集和验证集并绘图；在线训练指标另外记录为 online_loss/online_accuracy，二者不可混用。epoch=0 表示训练前。

训练 loader 使用独立随机 generator，每次完整实验重建。snapshot 复制参数，避免后续更新覆盖“最佳模型”。训练结束加载验证集选出的参数；返回模型可能来自最后一轮以前。


In [ ]:
def fit(kind, lr=0.1, epochs=5, seed=42):
    model = build_model(kind, seed)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loader = make_loader(train_set, BATCH_SIZE, shuffle=True, seed=ORDER_SEED)
    history = []
    best_key, best_state, best_epoch = None, None, None
    started = time.perf_counter()
    for epoch in range(epochs + 1):
        online = None
        if epoch > 0:
            online = train_epoch(model, loader, optimizer, criterion, DEVICE)
        train_metrics = evaluate(model, train_eval_loader, criterion, DEVICE)
        val_metrics = evaluate(model, val_loader, criterion, DEVICE)
        row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "seconds": time.perf_counter() - started,
            "online_loss": None if online is None else online["loss"],
            "online_accuracy": None if online is None else online["accuracy"],
        }
        history.append(row)
        key = (val_metrics["accuracy"], -val_metrics["loss"])
        if best_key is None or key > best_key:
            best_key, best_state, best_epoch = key, snapshot(model), epoch
        print(
            f"{kind} epoch={epoch}: train loss={row['train_loss']:.4f}, "
            f"acc={row['train_accuracy']:.2%}; val loss={row['val_loss']:.4f}, "
            f"acc={row['val_accuracy']:.2%}; elapsed={row['seconds']:.1f}s",
            flush=True,
        )
    model.load_state_dict(best_state)
    model.eval()
    return {
        "model": model,
        "history": history,
        "best_epoch": best_epoch,
        "best_val_accuracy": best_key[0],
        "best_val_loss": -best_key[1],
        "parameters": sum(p.numel() for p in model.parameters()),
        "seconds": time.perf_counter() - started,
        "kind": kind,
        "lr": lr,
        "epochs": epochs,
        "seed": seed,
        "order_seed": ORDER_SEED,
        "split_seed": SPLIT_SEED,
    }

In [ ]:
mlp_run = fit("mlp", lr=LR, epochs=EPOCHS, seed=SEED)
assert len(mlp_run["history"]) == EPOCHS + 1
restored_val = evaluate(mlp_run["model"], val_loader, criterion, DEVICE)
assert abs(restored_val["accuracy"] - mlp_run["best_val_accuracy"]) < 1e-12
assert abs(restored_val["loss"] - mlp_run["best_val_loss"]) < 1e-6
_ = plot_history({"mlp": mlp_run["history"]})
print("selected MLP epoch:", mlp_run["best_epoch"])

## Q6 · 线性基线与模型选择（必做）

**运行前写预测：** 与单隐藏层网络相比，线性分类器的训练速度、参数量、训练和验证准确率会怎样？说明依据。

对照固定划分、预处理、学习率、优化器、batch size、epoch 数和批次顺序。每次独立初始化；不同形状的模型不具有“相同初始参数”。主模型与基线同时改变了非线性与参数量，只能讨论这两种结构在当前训练预算下的表现。

补全结果表，并根据验证集选择最终模型。测试集在这里仍不使用。


In [ ]:
linear_run = fit("linear", lr=LR, epochs=EPOCHS, seed=SEED)
runs = {"linear": linear_run, "mlp": mlp_run}
_ = plot_history({name: run["history"] for name, run in runs.items()})
print("model | parameters | selected epoch | train acc | val acc | val loss | seconds")
for name, run in runs.items():
    best_row = run["history"][run["best_epoch"]]
    print(
        f"{name} | {run['parameters']} | {run['best_epoch']} | "
        f"{best_row['train_accuracy']:.4f} | {run['best_val_accuracy']:.4f} | "
        f"{run['best_val_loss']:.4f} | {run['seconds']:.1f}"
    )
# 同一规则在模型之间选择；完全相同按字典顺序优先选更简单的 linear。
selected_name = max(
    runs,
    key=lambda name: (runs[name]["best_val_accuracy"], -runs[name]["best_val_loss"]),
)
selected_run = runs[selected_name]
final_model = selected_run["model"]
print("FROZEN choice:", selected_name, "epoch:", selected_run["best_epoch"])

**我的对照记录：**

| 模型   | 参数量 | 选定轮次 | 训练准确率 | 验证准确率 | 验证损失 | 耗时 |
| ------ | ------ | -------- | ---------- | ---------- | -------- | ---- |
| linear | 待填   | 待填     | 待填       | 待填       | 待填     | 待填 |
| mlp    | 待填   | 待填     | 待填       | 待填       | 待填     | 待填 |

根据实际证据解释是否值得增加隐藏层，并写明结论的适用范围：待填写。


## Q7 · 最终测试与错分样本（必做）

先确认 Q6 已冻结模型、轮次和训练方案，再打开这一部分。只对最终所选模型报告测试指标，不根据测试集再修改学习率或结构。

教师提供按顺序收集前 12 张错分图像的工具。观察哪些数字容易混淆，区分“看到了什么”和“猜测为什么”。p(pred) 是模型给预测类别的 softmax 概率，不是经过校准的正确率保证。

思考：高准确率能否证明模型对任意手写数字都可靠？查看错分图像后提出改进，应该在什么数据上验证？


In [ ]:
test_set = load_test(download=DOWNLOAD)
assert len(test_set) == 10000
test_loader = make_loader(test_set, BATCH_SIZE)
final_train = evaluate(final_model, train_eval_loader, criterion, DEVICE)
final_val = evaluate(final_model, val_loader, criterion, DEVICE)
final_test = evaluate(final_model, test_loader, criterion, DEVICE)
print("final model:", selected_name, "epoch:", selected_run["best_epoch"])
print("train:", final_train, "validation:", final_val, "TEST:", final_test)
print("test errors:", round((1 - final_test["accuracy"]) * final_test["n"]))
errors = collect_errors(final_model, test_loader, DEVICE, limit=12)
if errors is not None:
    error_images, error_labels, error_preds, error_probs = errors
    _ = plot_samples(
        error_images,
        error_labels,
        error_preds,
        error_probs,
        title="First test errors (fixed order)",
    )
else:
    print("本次测试没有错分样本；不构造虚假的错误图。")

**我的解释与证据：** 待填写。


## Q8 · 学习率对照（选做）

固定 MLP、同样初始化、数据划分、批次顺序和 epoch 数，仅改变学习率为 0.01、0.1、0.5。已有 0.1 的结果可直接复用；其他值从新模型开始，不能在 final_model 上继续训练。

把 RUN_EXTENSION 改为 True 前，写下对速度与稳定性的预测。只使用训练／验证指标，不再次查看测试集。若出现非有限值，记录学习率、出错轮次和现象并停止该次实验，不将中断曲线称为收敛。


In [ ]:
RUN_EXTENSION = False
if RUN_EXTENSION:
    lr_runs = {"lr=0.1": mlp_run}
    for trial_lr in [0.01, 0.5]:
        try:
            lr_runs[f"lr={trial_lr}"] = fit(
                "mlp", lr=trial_lr, epochs=EPOCHS, seed=SEED
            )
        except FloatingPointError as error:
            print("stopped learning rate:", trial_lr, "reason:", error)
    _ = plot_history({name: run["history"] for name, run in lr_runs.items()})
else:
    print("Q8 skipped (optional); Q1–Q7 are unaffected")

**我的解释与证据：** 待填写。


**我的解释与证据：** 待填写。


## 参考资料

本实验的题目、检查和文字解释为课程自拟，参考下列公开材料的教学结构：

- [d2l 3.5：图像分类数据集](https://zh.d2l.ai/chapter_linear-networks/image-classification-dataset.html)（原书使用 Fashion-MNIST，本实验采用 MNIST）。
- [d2l 3.7：softmax 回归的简洁实现](https://zh.d2l.ai/chapter_linear-networks/softmax-regression-concise.html)。
- [d2l 4.3：多层感知机的简洁实现](https://zh.d2l.ai/chapter_multilayer-perceptrons/mlp-concise.html)。
- [PyTorch：优化模型参数](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)。
- [torchvision：MNIST](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.MNIST.html)。

本实验另设验证集，训练与评估代码显式展开，不依赖 d2l 的训练封装。
